In [ ]:
!pip install -q transformers datasets torch scikit-learn wandb

import pandas as pd
import numpy as np
import torch
import wandb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from google.colab import drive

In [ ]:
# 1. Kết nối Google Drive và Đọc dữ liệu
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.tsv', sep='\t')
df = df[['Phrase', 'Sentiment']]

Mounted at /content/drive


In [ ]:
# 2. Chia tập Train và Validation
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
# 3. Tokenization
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['Phrase'], padding='max_length', truncation=True, max_length=64)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("Sentiment", "labels").remove_columns(['Phrase', '__index_level_0__'])
val_dataset = val_dataset.rename_column("Sentiment", "labels").remove_columns(['Phrase', '__index_level_0__'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/140454 [00:00<?, ? examples/s]

Map:   0%|          | 0/15606 [00:00<?, ? examples/s]

In [ ]:
# 4. KHỞI TẠO WEIGHTS & BIASES
wandb.login()
wandb.init(
    entity="trungdunglebui17112004-ho-chi-minh-city-university-of-te",
    project="sentiment-analysis-on-movie-reviews",
    group="DeepLearning_BERT",
    name="Bert-LR-2e5-Run"
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hieu-nguyen2310959 (trungdunglebui17112004-ho-chi-minh-city-university-of-te) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:

# 5. CẤU HÌNH MÔ HÌNH VÀ TRAINING ARGUMENTS (Đã thêm Learning Rate)
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5)

training_args = TrainingArguments(
    output_dir='./results',
    report_to="wandb",
    learning_rate=2e-5,
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# 6. ĐỊNH NGHĨA HÀM TÍNH TOÁN

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Tính toán Accuracy và F1-Score
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')

    try:
        wandb.log({
            "confusion_matrix": wandb.plot.confusion_matrix(
                preds=predictions,
                y_true=labels,
                class_names=["0 (Neg)", "1 (Somewhat Neg)", "2 (Neu)", "3 (Somewhat Pos)", "4 (Pos)"]
            )
        })
    except Exception as e:
        print(f"Lỗi vẽ Confusion Matrix: {e}")

    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

In [ ]:

# 7. KHỞI TẠO TRAINER VÀ HUẤN LUYỆN
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Bắt đầu học
trainer.train()

# Đánh giá cuối cùng
trainer.evaluate()

wandb.finish()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.711682,0.719020,0.704216,0.613547,0.701231
2,0.650899,0.710242,0.704793,0.637798,0.704449
3,0.571140,0.726105,0.706844,0.635640,0.705498
4,0.512903,0.774242,0.702614,0.637645,0.703019


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

eval/accuracy,▄▅█▁▄
eval/f1_macro,▁█▇██
eval/f1_weighted,▁▆█▄▆
eval/loss,▂▁▃█▁
eval/runtime,█▅▂▁▇
eval/samples_per_second,▁▄▇█▂
eval/steps_per_second,▁▄▇█▂
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
train/global_step,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,▆▇▆▃▃▂▄▃▂▃█▂▃▂▃▂▄▅▂▁▅▅▅▂▃▅▄▅▇▆▅▆█▄▃▃▄▂▅▅
+2,...
